In [57]:
import pandas as pd

In [58]:
df = pd.read_excel('../data/dcInbox/dcinbox_export_116_b2.xlsx')
unnamed_cols = df.columns.str.contains('^Unnamed')
df = df.loc[:, ~unnamed_cols].copy()

In [59]:
def process_lexicon(dataframe, lexicon):
    """Process thematic lexicon and return theme and phrase results"""
    
    # Combine subject and body
    df_copy = dataframe.copy()
    df_copy['full_text'] = (df_copy['Subject'].fillna('').astype(str) + ' ' + 
                            df_copy['Body'].fillna('').astype(str)).str.lower()
    
    phrase_results = []
    theme_results = []
    
    for theme, phrases in lexicon.items():
        # Track which emails mention this theme (any phrase)
        theme_mask = pd.Series([False] * len(df_copy), index=df_copy.index)
        
        for phrase in phrases:
            # Count emails containing this specific phrase
            phrase_mask = df_copy['full_text'].str.contains(phrase, case=False, regex=False)
            count = phrase_mask.sum()
            percentage = (count / len(df_copy)) * 100
            
            phrase_results.append({
                'theme': theme,
                'phrase': phrase,
                'email_count': count,
                'percentage': percentage
            })
            
            # Add to theme mask
            theme_mask = theme_mask | phrase_mask
        
        # Calculate theme-level coverage
        theme_count = theme_mask.sum()
        theme_percentage = (theme_count / len(df_copy)) * 100
        
        theme_results.append({
            'theme': theme,
            'email_count': theme_count,
            'percentage': theme_percentage,
            'num_phrases': len(phrases)
        })
    
    return theme_results, phrase_results

In [60]:
"""
Policy Phrase Coverage Analysis - Theme-Based Version
Creates a thematic lexicon and calculates coverage at both phrase and theme levels
"""

# Group related phrases by policy theme
THEMATIC_LEXICON = {
    'Healthcare': [
        'health care',
        'healthcare',
        'public health',
        'mental health',
        'affordable care act',
        'obamacare',
        'covid',
        'covid-19',
        'covid-19 pandemic',
        'covid-19 vaccine',
        'covid-19 vaccination',
    ],
    'Voting Rights': [
        'voting rights',
        'voting rights act',
        'election security',
    ],
    'Gun Policy': [
        'gun violence',
        'gun safety',
        'gun control',
        'gun reform',
    ],
    'Social Safety Net': [
        'social security',
        'child care',
        'minimum wage',
    ],
    'Civil Rights': [
        'civil rights',
        'women\'s rights',
    ],
    'Reproductive Rights': [
        'reproductive rights',
        'abortion rights',
        'abortion access',
        'abortion care',
    ],
    'Climate & Energy': [
        'climate change',
        'renewable energy',
        'clean energy',
        'green energy',
    ],
    'Immigration': [
        'immigration reform',
        'immigration policy',
        'border security',
    ],
    'Criminal Justice': [
        'criminal justice',
        'police reform',
        'criminal justice reform',
    ],
}

dem_df = df[df['Party'] == 'Democrat'].copy()
print(f"Democrat emails: {len(dem_df)}")

theme_results, phrase_results = process_lexicon(dem_df, THEMATIC_LEXICON)


# Create results dataframes
phrase_df = pd.DataFrame(phrase_results).sort_values('percentage', ascending=False)
theme_df = pd.DataFrame(theme_results).sort_values('percentage', ascending=False)

# Display theme-level summary
print("\n" + "="*70)
print("THEME COVERAGE (emails mentioning ANY phrase in theme)")
print("="*70)

for i, row in enumerate(theme_df.itertuples(), 1):
    print(f"{i:2d}. {row.theme:25s}: {row.email_count:5d} emails ({row.percentage:5.2f}%) [{row.num_phrases} phrases]")

# Display top individual phrases
print("\n" + "="*70)
print("TOP INDIVIDUAL PHRASES BY COVERAGE")
print("="*70)

for i, row in enumerate(phrase_df.head(20).itertuples(), 1):
    print(f"{i:2d}. {row.phrase:30s} ({row.theme:20s}): {row.email_count:5d} ({row.percentage:5.2f}%)")

Democrat emails: 14693

THEME COVERAGE (emails mentioning ANY phrase in theme)
 1. Healthcare               :  8818 emails (60.01%) [11 phrases]
 2. Social Safety Net        :  2376 emails (16.17%) [3 phrases]
 3. Climate & Energy         :  1143 emails ( 7.78%) [4 phrases]
 4. Civil Rights             :   738 emails ( 5.02%) [2 phrases]
 5. Gun Policy               :   699 emails ( 4.76%) [4 phrases]
 6. Voting Rights            :   567 emails ( 3.86%) [3 phrases]
 7. Immigration              :   340 emails ( 2.31%) [3 phrases]
 8. Criminal Justice         :   336 emails ( 2.29%) [3 phrases]
 9. Reproductive Rights      :    80 emails ( 0.54%) [4 phrases]

TOP INDIVIDUAL PHRASES BY COVERAGE
 1. covid                          (Healthcare          ):  5773 (39.29%)
 2. covid-19                       (Healthcare          ):  5648 (38.44%)
 3. health care                    (Healthcare          ):  3752 (25.54%)
 4. public health                  (Healthcare          ):  3033 (20.64%)
 5.

In [ ]:
"""
Capture Full Phrase Matches with Email Metadata
Creates a detailed dataframe showing which emails contain which phrases,
including the actual matched text and email metadata
TO-DO: This should be combined with the process_lexicon function
"""

import re
from collections import defaultdict

# Create full_text column for dem_df
dem_df['full_text'] = (dem_df['Subject'].fillna('').astype(str) + ' ' + 
                       dem_df['Body'].fillna('').astype(str)).str.lower()

def extract_phrase_contexts(text, phrase, context_chars=50):
    """
    Extract the actual phrase matches with surrounding context
    """
    contexts = []
    # Use case-insensitive search
    pattern = re.compile(re.escape(phrase), re.IGNORECASE)
    
    for match in pattern.finditer(text):
        start = max(0, match.start() - context_chars)
        end = min(len(text), match.end() + context_chars)
        context = text[start:end]
        
        # Highlight the matched phrase
        highlighted = context.replace(match.group(), f"**{match.group()}**")
        contexts.append(highlighted)
    
    return contexts

phrase_matches = []

for theme, phrases in THEMATIC_LEXICON.items():
    for phrase in phrases:
        # Find emails containing this phrase
        phrase_mask = dem_df['full_text'].str.contains(phrase, case=False, regex=False)
        matching_emails = dem_df[phrase_mask]
        
        for idx, email_row in matching_emails.iterrows():
            # Extract contexts for this phrase in this email
            contexts = extract_phrase_contexts(email_row['full_text'], phrase)
            
            for context in contexts:
                phrase_matches.append({
                    'email_index': idx,
                    'theme': theme,
                    'phrase': phrase,
                    'matched_context': context,
                    'subject': email_row['Subject'],
                    'date': email_row.get('Date', ''),
                    'party': email_row['Party'],
                    'full_text_length': len(email_row['full_text'])
                })

# Create the detailed matches dataframe
matches_df = pd.DataFrame(phrase_matches)

# Count matches by theme
theme_counts = matches_df.groupby('theme').agg({
    'phrase': 'count',
    'email_index': 'nunique'
}).rename(columns={'phrase': 'total_matches', 'email_index': 'unique_emails'})



# store metadata about matches
for theme in matches_df['theme'].unique():
    theme_matches = matches_df[matches_df['theme'] == theme]



In [66]:
theme_matches.head(5)

,email_index,theme,phrase,matched_context,subject,date,party,full_text_length
92621,82,Criminal Justice,criminal justice,"mmigrants, dreamers, and refugees pass meaning...",SURVEY: Your Priorities for the 117th Congress,,Democrat,2873
92622,259,Criminal Justice,criminal justice,th disabilities -- out of classrooms and into ...,Senator Bennet's Weekly Update,,Democrat,4408
92623,321,Criminal Justice,criminal justice,here at home. addressing police brutality and ...,A Note from Congresswoman Vel√°zquez,,Democrat,20780
92624,321,Criminal Justice,criminal justice,scriminatory and racist policies within americ...,A Note from Congresswoman Vel√°zquez,,Democrat,20780
92625,321,Criminal Justice,criminal justice,"unding support, with the endorsement of dozens...",A Note from Congresswoman Vel√°zquez,,Democrat,20780


In [68]:
"""
Republican Policy Phrase Coverage Analysis - Theme-Based Lexicon
"""

REPUBLICAN_THEMATIC_LEXICON = {
    'Border & Immigration': [
        'border',
        'southern border',
        'border security',
        'border crisis',
        'illegal immigration',
        'immigration enforcement',
        'secure the border',
        'border wall',
        'sanctuary cities',
        'catch and release',
    ],
    
    'Election Integrity': [
        'election integrity',
        'election security',
        'voter fraud',
        'voter id',
        'election reform',
        'ballot harvesting',
        'mail-in voting',
        'mail-in ballot',
        'voter verification',
    ],
    
    'Biden Administration Critique': [
        'biden',
        'president biden',
        'biden administration',
        'biden harris',
        'failed policies',
        'biden agenda',
        'radical left',
    ],
    
    'Trump & MAGA': [
        'trump',
        'president trump',
        'trump administration',
        'make america great',
        'maga',
        'america first',
    ],
    
    'Law Enforcement & Crime': [
        'law enforcement',
        'police',
        'law and order',
        'crime',
        'violent crime',
        'defund the police',
        'back the blue',
        'public safety',
        'criminal justice',
    ],
    
    'National Security & Defense': [
        'national security',
        'homeland security',
        'defense',
        'military',
        'veterans',
        'armed forces',
        'defense spending',
        'national defense',
    ],
    
    'Economy & Taxes': [
        'taxes',
        'tax cuts',
        'tax relief',
        'tax reform',
        'economy',
        'inflation',
        'jobs',
        'economic growth',
        'small business',
        'regulations',
        'government spending',
        'national debt',
        'deficit',
    ],
    
    'Energy & Climate': [
        'energy independence',
        'energy production',
        'oil and gas',
        'fossil fuels',
        'pipeline',
        'energy prices',
        'gas prices',
        'green new deal',
    ],
    
    'Second Amendment': [
        'second amendment',
        '2nd amendment',
        'gun rights',
        'right to bear arms',
        'gun control',
        'gun grab',
        'gun confiscation',
    ],
    
    'Social Issues & Values': [
        'religious freedom',
        'religious liberty',
        'life',
        'pro-life',
        'unborn',
        'sanctity of life',
        'family values',
        'traditional values',
        'parental rights',
    ],
    
    'Education & CRT': [
        'education',
        'school choice',
        'parental rights',
        'critical race theory',
        'crt',
        'woke',
        'indoctrination',
        'curriculum',
    ],
    
    'Government Overreach': [
        'big government',
        'government overreach',
        'bureaucracy',
        'federal government',
        'mandates',
        'federal overreach',
        'states rights',
        'freedom',
        'liberty',
    ],
    
    'Healthcare': [
        'obamacare',
        'affordable care act',
        'healthcare',
        'health care',
        'medicare',
        'medicaid',
        'healthcare costs',
    ],
    
    'COVID Policy': [
        'covid',
        'covid-19',
        'pandemic',
        'lockdowns',
        'vaccine mandates',
        'mask mandates',
        'covid restrictions',
        'covid response',
    ],
    
    'Social Security & Entitlements': [
        'social security',
        'medicare',
        'medicaid',
        'entitlements',
        'welfare',
    ],
    
    'Congress & Legislation': [
        'legislation',
        'this bill',
        'the bill',
        'house',
        'senate',
        'congress',
        'committee',
        'vote',
        'law',
    ],
    
    'Supreme Court & Judiciary': [
        'supreme court',
        'judicial',
        'judges',
        'department of justice',
        'courts',
        'constitutional',
    ],
    
    'Big Tech & Censorship': [
        'big tech',
        'censorship',
        'social media',
        'free speech',
        'first amendment',
        'cancel culture',
        'silicon valley',
    ],
}

rep_df = df[df['Party'] == 'Republican'].copy()
print(f"Republican emails: {len(rep_df)}")

# Combine subject and body
print("\nCombining subject and body text...")
rep_df['full_text'] = (rep_df['Subject'].fillna('').astype(str) + ' ' + 
                        rep_df['Body'].fillna('').astype(str)).str.lower()

theme_results, phrase_results = process_lexicon(rep_df, REPUBLICAN_THEMATIC_LEXICON)


# Create results dataframes
phrase_df = pd.DataFrame(phrase_results).sort_values('percentage', ascending=False)
theme_df = pd.DataFrame(theme_results).sort_values('percentage', ascending=False)

# Display theme-level summary
print("\n" + "="*70)
print("THEME COVERAGE (emails mentioning ANY phrase in theme)")
print("="*70)

for i, row in enumerate(theme_df.itertuples(), 1):
    print(f"{i:2d}. {row.theme:35s}: {row.email_count:5d} emails ({row.percentage:5.2f}%) [{row.num_phrases} phrases]")

# Display top individual phrases
print("\n" + "="*70)
print("TOP INDIVIDUAL PHRASES BY COVERAGE")
print("="*70)

for i, row in enumerate(phrase_df.head(30).itertuples(), 1):
    print(f"{i:2d}. {row.phrase:30s} ({row.theme:30s}): {row.email_count:5d} ({row.percentage:5.2f}%)")

Republican emails: 16136

Combining subject and body text...

THEME COVERAGE (emails mentioning ANY phrase in theme)
 1. Congress & Legislation             : 15605 emails (96.71%) [9 phrases]
 2. Economy & Taxes                    :  8748 emails (54.21%) [13 phrases]
 3. National Security & Defense        :  7535 emails (46.70%) [8 phrases]
 4. Trump & MAGA                       :  6515 emails (40.38%) [6 phrases]
 5. COVID Policy                       :  5913 emails (36.64%) [8 phrases]
 6. Healthcare                         :  5160 emails (31.98%) [7 phrases]
 7. Social Issues & Values             :  4997 emails (30.97%) [9 phrases]
 8. Government Overreach               :  4844 emails (30.02%) [9 phrases]
 9. Law Enforcement & Crime            :  3208 emails (19.88%) [9 phrases]
10. Education & CRT                    :  3132 emails (19.41%) [8 phrases]
11. Social Security & Entitlements     :  2531 emails (15.69%) [5 phrases]
12. Big Tech & Censorship              :  2436 emails (15